In [1]:
import pandas as pd
import glob
import os
import json
import re
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

In [2]:
results_dir = "offline_evaluation_results"
results_output_dir = "analysis_results"

In [3]:
def load_all_metrics(results_dir="results"):
    """
    Iterates through all JSONL files in the results directory, extracts parameters
    from filenames, loads their contents, and returns a combined DataFrame.
    """
    pattern = re.compile(
        r"confidence_metrics_(?P<model>.+?)_(?P<turn>\d+)_top(?P<topk>\d+)logprobs_tail(?P<tail>\d+)_group(?P<group>\d+)\.jsonl"
    )

    all_dfs = []

    for filename in os.listdir(results_dir):
        if not filename.endswith(".jsonl"):
            continue

        match = pattern.match(filename)
        if not match:
            continue  # skip files that don't fit the pattern

        params = match.groupdict()
        file_path = os.path.join(results_dir, filename)

        # Load JSONL file
        try:
            df = pd.read_json(file_path, lines=True)
        except ValueError:
            print(f"⚠️ Skipping malformed file: {filename}")
            continue
        
        if "response_text" in df.columns:
            df = df.drop(columns=["response_text"])

        # Add parameters as columns
        df["model"] = params["model"]
        df["turn"] = int(params["turn"])
        df["top_k_logprobs"] = int(params["topk"])
        df["tail_n"] = int(params["tail"])
        df["group_size"] = int(params["group"])

        all_dfs.append(df)

    if not all_dfs:
        print("No valid JSONL files found.")
        return pd.DataFrame()

    return pd.concat(all_dfs, ignore_index=True)


In [4]:
df = load_all_metrics(results_dir)

In [5]:
df.head()

,problem_id,rollout_idx,extracted_answer,is_correct,expected_answer,total_tokens,mean_full,mean_tail,mean_group_lowest,mean_group_bottom_pct,...,gap_probs_tail,gap_probs_group_lowest,gap_probs_group_bottom_pct,gap_probs_group_highest,gap_probs_group_top_pct,model,turn,top_k_logprobs,tail_n,group_size
0,aime_2025_5,0,504,True,504,4406,16.137308,14.900967,9.422558,13.513722,...,0.827562,0.806519,0.811453,0.999691,0.897176,Qwen8b,1,8,2048,1024
1,aime_2025_5,1,504,True,504,4732,15.954173,14.829415,9.422558,13.347687,...,0.832866,0.785139,0.790034,0.999691,0.889084,Qwen8b,1,8,2048,1024
2,aime_2025_5,2,504,True,504,4496,15.980498,15.806880,9.422558,14.102894,...,0.841437,0.777922,0.784769,0.999691,0.886770,Qwen8b,1,8,2048,1024
3,aime_2025_5,3,504,True,504,5760,16.190179,14.782489,9.422558,13.467209,...,0.839204,0.763011,0.795337,0.999691,0.924077,Qwen8b,1,8,2048,1024
4,aime_2025_5,4,504,True,504,6171,15.324086,13.994901,9.422558,13.055213,...,0.813131,0.738249,0.755648,0.999696,0.885076,Qwen8b,1,8,2048,1024


In [6]:
df = df[df.turn.isin([1, 2, 3])]
len(df)

63360

In [7]:
df["is_correct"] = df["is_correct"].astype(int)

In [8]:
def compute_correlation_stats(data, correct_label='is_correct'):
    """
    Compute mean and std correlation of each metric with correctness across turns.
    """
    numeric_cols = [
        col for col in data.columns
        if data[col].dtype in ("float64", "int64") and col not in ("is_correct", "turn")
    ]

    turns = sorted(data["turn"].dropna().unique())
    per_turn_corrs = []
    for turn in turns:
        df_turn = data[data["turn"] == turn]
        if df_turn.empty:
            continue
        corr_per_turn = {
            col: df_turn[col].corr(df_turn[correct_label])
            for col in numeric_cols
        }
        per_turn_corrs.append(pd.Series(corr_per_turn, name=f"turn{turn}"))

    per_turn_corrs_df = pd.concat(per_turn_corrs, axis=1)
    mean_corr = per_turn_corrs_df.mean(axis=1)
    std_corr = per_turn_corrs_df.std(axis=1)

    summary = pd.DataFrame({
        "Metric": mean_corr.index,
        "Mean_Correlation": mean_corr.values,
        "Std_Correlation": std_corr.values
    })
    return summary


def plot_correlation_heatmap_matplotlib(data, correct_label='is_correct', metrics=None):
    # Step 1: compute correlations
    model = data["model"].iloc[0]
    corr_summary = compute_correlation_stats(data, correct_label)

    # Step 2: define subsets and metric types
    subsets = ["full", "tail", "group_lowest", "group_highest", "group_bottom_pct", "group_top_pct"]
    metric_types = metrics

    # Step 3: initialize matrices
    heatmap_mean = np.full((len(subsets), len(metric_types)), np.nan)
    heatmap_std = np.full((len(subsets), len(metric_types)), np.nan)

    # Step 4: fill matrices
    for i, subset in enumerate(subsets):
        for j, metric in enumerate(metric_types):
            col_name = f"{metric}_{subset}"
            match = corr_summary[corr_summary["Metric"] == col_name]
            if not match.empty:
                heatmap_mean[i, j] = match["Mean_Correlation"].values[0]
                heatmap_std[i, j] = match["Std_Correlation"].values[0]

    # Step 5: plot heatmap
    fig, ax = plt.subplots(figsize=(10, 6))
    im = ax.imshow(heatmap_mean, cmap="coolwarm", aspect="auto", vmin=-1, vmax=1)

    # Colorbar
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label("Mean correlation with correctness", rotation=270, labelpad=15)

    # Axes setup
    ax.set_xticks(np.arange(len(metric_types)))
    ax.set_yticks(np.arange(len(subsets)))
    ax.set_xticklabels(metric_types, rotation=45, ha="right")
    ax.set_yticklabels(subsets)
    ax.set_title(f"Correlation of Confidence Metrics with Correctness ({correct_label}) - model: {model}")

    # Annotate cells with mean ± std
    for i in range(len(subsets)):
        for j in range(len(metric_types)):
            mean_val = heatmap_mean[i, j]
            std_val = heatmap_std[i, j]
            if not np.isnan(mean_val):
                ax.text(
                    j, i,
                    f"{mean_val:.2f}±{std_val:.2f}",
                    ha="center",
                    va="center",
                    color="black",
                    fontsize=8
                )

    plt.tight_layout()
    return fig




In [9]:
import numpy as np
from scipy.spatial.distance import jensenshannon

def compute_js_divergence(correct_vals, incorrect_vals, bins):
    """
    Compute Jensen-Shannon divergence between two distributions.
    """
    if len(correct_vals) == 0 or len(incorrect_vals) == 0:
        return np.nan
    
    # Create histograms with the same bins
    correct_hist, _ = np.histogram(correct_vals, bins=bins, density=True)
    incorrect_hist, _ = np.histogram(incorrect_vals, bins=bins, density=True)
    
    # Normalize to ensure they sum to 1 (probability distributions)
    correct_hist = correct_hist / (correct_hist.sum() + 1e-10)
    incorrect_hist = incorrect_hist / (incorrect_hist.sum() + 1e-10)
    
    # Add small epsilon to avoid log(0)
    epsilon = 1e-10
    correct_hist = correct_hist + epsilon
    incorrect_hist = incorrect_hist + epsilon
    
    # Re-normalize after adding epsilon
    correct_hist = correct_hist / correct_hist.sum()
    incorrect_hist = incorrect_hist / incorrect_hist.sum()
    
    # Compute JS divergence
    js_div = jensenshannon(correct_hist, incorrect_hist, base=2)
    
    return js_div

def plot_correlation_barplot(data, title="", subplot_fontsize=20, correct_label='is_correct'):
    """
    Plot mean ± std correlation per metric as a bar chart with JS divergence.
    """
    numeric_cols = [
        col for col in data.columns
        if data[col].dtype in ("float64", "int64") and col != "is_correct" and col != "turn" and col != 'is_majority_correct'
    ]
    num_plots = len(numeric_cols) + 1
    cols = 6
    rows = (num_plots // cols) 

    fig, axes = plt.subplots(nrows=rows, ncols=cols, figsize=(40, 5 * rows))
    axes = axes.flatten()

    # Define colors and opacity
    color_correct = 'green'
    color_incorrect = 'red'
    alpha_val = 0.6
    bins_count = 30

    # Plot each numeric column with shared bins
    for i, col in enumerate(numeric_cols):
        correct_vals = data.loc[data[correct_label] == 1, col].dropna()
        incorrect_vals = data.loc[data[correct_label] == 0, col].dropna()
        
        # Compute shared bin edges from the combined data
        combined_vals = data[col].dropna()
        bins = np.histogram_bin_edges(combined_vals, bins=bins_count)
        
        # Compute JS divergence
        js_div = compute_js_divergence(correct_vals, incorrect_vals, bins)
        
        # Plot incorrect (behind)
        axes[i].hist(incorrect_vals, bins=bins, color=color_incorrect, edgecolor='black',
                    alpha=alpha_val, label='Incorrect')
        # Plot correct (on top)
        axes[i].hist(correct_vals, bins=bins, color=color_correct, edgecolor='black',
                    alpha=alpha_val, label='Correct')
        
        # Add JS divergence to title
        axes[i].set_title(f"{col}\nJS Div: {js_div:.4f}", fontsize=subplot_fontsize)
        axes[i].set_xlabel("Value")
        axes[i].set_ylabel("Frequency", fontsize=15)
        axes[i].tick_params(axis='x', labelsize=16)
        axes[i].tick_params(axis='y', labelsize=16)
        axes[i].legend()
        #axes[i].set_ylim(0, 200)

    fig.suptitle(f"Distribution of Metrics by Correctness - {title}", fontsize=30, y=1.02)
    
    # Hide unused subplots
    for ax in axes[len(numeric_cols):]:
        ax.axis("off")

    plt.tight_layout()
    return fig

In [10]:
standard_confidence_metrics = [
    "mean_full",
    "median_full",
    "variance_full",
    "gap_full",
    'gap_probs_full',
    #"distinctiveness_full",
    'entropy_full',
    #'fisher_rao_distance_full',
    #'kl_divergence_full',
    #'inverse_probability_full',
    #'perplexity_full',
    #'renyi_divergence_full',
    #'top_prob_full'
    ]

tail_confidence_metrics = [
    "mean_tail",
    "median_tail",
    "variance_tail",
    "gap_tail",
    "gap_probs_tail",
    #"distinctiveness_tail",
    'entropy_tail',
    #'fisher_rao_distance_tail',
    #'kl_divergence_tail',
    #'inverse_probability_tail',
    #'perplexity_tail',
    #'renyi_divergence_tail',
    #'top_prob_tail'
    ]

group_lowest_metrics = [
    'mean_group_lowest',
    'median_group_lowest',
    'variance_group_lowest',
    'gap_group_lowest',
    'gap_probs_group_lowest',
    #'distinctiveness_group_lowest',
    'entropy_group_lowest',
    #'fisher_rao_distance_group_lowest',
    #'kl_divergence_group_lowest',
    #'inverse_probability_group_lowest',
    #'perplexity_group_lowest',
    #'renyi_divergence_group_lowest',
    #'top_prob_group_lowest'
]

group_highest_metrics = [
    'mean_group_highest',
    'median_group_highest',
    'variance_group_highest',
    'gap_group_highest',
    'gap_probs_group_highest',
    #'distinctiveness_group_highest',
    'entropy_group_highest',
    #'fisher_rao_distance_group_highest',
    #'kl_divergence_group_highest',
    #'inverse_probability_group_highest',
    #'perplexity_group_highest',
    #'renyi_divergence_group_highest',
    #'top_prob_group_highest'
]

group_bottom_metrics = [
    'mean_group_bottom_pct',
    'median_group_bottom_pct',
    'variance_group_bottom_pct',
    'gap_group_bottom_pct',
    'gap_probs_group_bottom_pct',
    #'distinctiveness_group_bottom_pct',
    'entropy_group_bottom_pct',
    #'fisher_rao_distance_group_bottom_pct',
    #'kl_divergence_group_bottom_pct',
    #'inverse_probability_group_bottom_pct',
    #'perplexity_group_bottom_pct',
    #'renyi_divergence_group_bottom_pct',
    #'top_prob_group_bottom_pct'
]

group_top_metrics = [
    'mean_group_top_pct',
    'median_group_top_pct',
    'variance_group_top_pct',
    'gap_group_top_pct',
    'gap_probs_group_top_pct',
    #'distinctiveness_group_top_pct',
    'entropy_group_top_pct',
    #'fisher_rao_distance_group_top_pct',
    #'kl_divergence_group_top_pct',
    #'inverse_probability_group_top_pct',
    #'perplexity_group_top_pct',
    #'renyi_divergence_group_top_pct',
    #'top_prob_group_top_pct'
]

base_metrics = [
    'mean',
    'median',
    'variance',
    'gap',
    'gap_probs',
    #'distinctiveness',
    'entropy',
    #'fisher_rao_distance',
    #'kl_divergence',
    #'inverse_probability',
    #'perplexity',
    #'renyi_divergence',
    #'top_prob'
]

In [11]:
df_tail_2048_k10 = df[(df["tail_n"] == 2048) & (df["top_k_logprobs"] == 10) & (df["group_size"] == 1024) ]
len(df_tail_2048_k10)

5760

In [12]:


def plot_histograms_by_correctness(df, metrics, title="", figname=None):
    models = df["model"].unique()

    for model in models:
        os.makedirs(f"{results_output_dir}/figures/{model}", exist_ok=True)
        print(f"\nProcessing model: {model}")
        data_model = df[(df["model"] == model) ]
        data_model = data_model[list(metrics) + ["is_correct", "turn", 'model']]
        fig_bar = plot_correlation_barplot(data_model, title = title + " - " + model)
        fig_heatmap = plot_correlation_heatmap_matplotlib(data_model, metrics=base_metrics, correct_label='is_correct')
        
        fig_bar.savefig(f"{results_output_dir}/figures/{model}/{figname}_barplot.png", bbox_inches='tight')
        fig_heatmap.savefig(f"{results_output_dir}/figures/{model}/{figname}_heatmap.png", bbox_inches='tight')


In [13]:
all_metrics = standard_confidence_metrics + tail_confidence_metrics + group_lowest_metrics + group_highest_metrics + group_bottom_metrics + group_top_metrics 

In [ ]:
plot_histograms_by_correctness(df_tail_2048_k10, all_metrics, title="Tail = 2048, n_gen=64, top_logprobs=10", figname="tail2048_n64_top10")

In [14]:
def plot_grouped_histograms_with_js(
    data,
    group_col,
    title="",
    subplot_fontsize=18,
    correct_label="is_correct",
    bins_count=30,
):
    """
    Plot histograms of each numeric column grouped by unique values of group_col.
    Each unique value of group_col gets its own row of subplots.
    
    Parameters
    ----------
    data : pd.DataFrame
        The dataframe containing metric columns and a correctness label.
    group_col : str
        The column name to group by (e.g. 'tail_n', 'group_size', 'top_k_logprobs').
    title : str
        Title of the figure.
    correct_label : str
        The column that indicates correctness (1/0).
    bins_count : int
        Number of histogram bins.
    """

    # Validate group_col
    if group_col not in data.columns:
        raise ValueError(f"Column '{group_col}' not found in dataframe.")

    # Select numeric columns to plot (ignore metadata and grouping col)
    numeric_cols = [
        col for col in data.columns
        if data[col].dtype in ("float64", "int64")
        and col not in ['is_correct', 'is_majority_correct', 'turn', group_col]
    ]

    # Get sorted list of unique values in the grouping column
    group_values = sorted(data[group_col].unique())
    num_cols = len(numeric_cols)

    # Create subplots: one row per group value
    fig, axes = plt.subplots(
        nrows=len(group_values),
        ncols=num_cols,
        figsize=(6 * num_cols, 5 * len(group_values))
    )

    # Handle single-row/col edge cases
    if len(group_values) == 1:
        axes = np.array([axes])
    if num_cols == 1:
        axes = axes.reshape(len(group_values), 1)

    # Define colors
    color_correct = "green"
    color_incorrect = "red"
    alpha_val = 0.6

    # Loop through each group and each numeric metric
    for row_idx, gval in enumerate(group_values):
        df_group = data[data[group_col] == gval]

        for col_idx, col in enumerate(numeric_cols):
            ax = axes[row_idx, col_idx]

            correct_vals = df_group.loc[df_group[correct_label] == 1, col].dropna()
            incorrect_vals = df_group.loc[df_group[correct_label] == 0, col].dropna()

            if correct_vals.empty or incorrect_vals.empty:
                ax.axis("off")
                continue

            bins = np.histogram_bin_edges(df_group[col].dropna(), bins=bins_count)

            # Compute JS divergence
            js_div = compute_js_divergence(correct_vals, incorrect_vals, bins)

            # Plot histograms
            ax.hist(
                incorrect_vals, bins=bins,
                color=color_incorrect, edgecolor="black",
                alpha=alpha_val, label="Incorrect"
            )
            ax.hist(
                correct_vals, bins=bins,
                color=color_correct, edgecolor="black",
                alpha=alpha_val, label="Correct"
            )

            ax.set_title(f"{col}\n{group_col}={gval} | JS Div: {js_div:.4f}", fontsize=subplot_fontsize)
            ax.set_xlabel("Value")
            ax.set_ylabel("Frequency", fontsize=14)
            ax.tick_params(axis="x", labelsize=12)
            ax.tick_params(axis="y", labelsize=12)
            if row_idx == 0 and col_idx == 0:
                ax.legend()

    fig.suptitle(
        f"Distribution of Metrics by Correctness (Grouped by {group_col}) - {title}",
        fontsize=28,
        y=1.02
    )
    plt.tight_layout()
    return fig
    
def compute_group_correlation_stats(data, group_col, correct_label="is_correct"):
    """
    Compute mean and std correlation of each numeric metric with correctness,
    grouped by a chosen column (e.g., tail_n, group_size, top_k_logprobs, etc.).
    """
    numeric_cols = [
        col for col in data.columns
        if data[col].dtype in ("float64", "int64")
        and col not in ("is_correct", "turn", group_col)
    ]

    group_values = sorted(data[group_col].dropna().unique())
    per_group_corrs = []

    for g in group_values:
        df_group = data[data[group_col] == g]
        if df_group.empty:
            continue

        corr_per_group = {
            col: df_group[col].corr(df_group[correct_label])
            for col in numeric_cols
        }
        per_group_corrs.append(pd.Series(corr_per_group, name=f"{group_col}_{g}"))

    per_group_corrs_df = pd.concat(per_group_corrs, axis=1)
    mean_corr = per_group_corrs_df.mean(axis=1)
    std_corr = per_group_corrs_df.std(axis=1)

    summary = pd.DataFrame({
        "Metric": mean_corr.index,
        "Mean_Correlation": mean_corr.values,
        "Std_Correlation": std_corr.values
    })
    return summary


def plot_grouped_correlation_heatmap(
    data,
    group_col,
    correct_label="is_correct",
    metrics=None,
    title=""
):
    """
    Plot a heatmap where rows correspond to group_col values (e.g. tail_n),
    columns correspond to metric names, and cell color = mean correlation with correctness.
    """
    # Compute correlations
    corr_summary = compute_group_correlation_stats(data, group_col, correct_label)

    # Determine metrics (if not provided, infer them)
    if metrics is None:
        metrics = [
            m for m in corr_summary["Metric"].unique()
            if not m.startswith(correct_label)
        ]

    group_values = sorted(data[group_col].dropna().unique())

    # Initialize matrices
    heatmap_mean = np.full((len(group_values), len(metrics)), np.nan)
    heatmap_std = np.full((len(group_values), len(metrics)), np.nan)

    # Fill matrices
    for i, g in enumerate(group_values):
        df_group = data[data[group_col] == g]
        for j, metric in enumerate(metrics):
            val = df_group[metric].corr(df_group[correct_label])
            if not np.isnan(val):
                heatmap_mean[i, j] = val
                # std cannot be computed from single correlation — could skip or reuse std
                heatmap_std[i, j] = 0.0

    # Plot heatmap
    fig, ax = plt.subplots(figsize=(1.5 * len(metrics), 0.8 * len(group_values)+2))
    im = ax.imshow(heatmap_mean, cmap="coolwarm", aspect="auto", vmin=-1, vmax=1)

    # Colorbar
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label("Mean correlation with correctness", rotation=270, labelpad=15)

    # Axes setup
    ax.set_xticks(np.arange(len(metrics)))
    ax.set_yticks(np.arange(len(group_values)))
    ax.set_xticklabels(metrics, rotation=45, ha="right")
    ax.set_yticklabels(group_values)
    ax.set_title(f"Correlation of Metrics with Correctness (Grouped by {group_col})")

    # Annotate cells with correlation value
    for i in range(len(group_values)):
        for j in range(len(metrics)):
            mean_val = heatmap_mean[i, j]
            if not np.isnan(mean_val):
                ax.text(
                    j, i,
                    f"{mean_val:.2f}",
                    ha="center",
                    va="center",
                    color="black",
                    fontsize=8
                )

    plt.tight_layout()
    return fig



In [15]:
model = df['model'].unique()[0]

In [ ]:
df_tails = df[df["top_k_logprobs"].isin([10]) & df['group_size'].isin([1024])][tail_confidence_metrics + ['tail_n', 'is_correct', 'turn', 'model']]


fig_hist_tails = plot_grouped_histograms_with_js(df_tails, group_col='tail_n')
fig_corr_tails = plot_grouped_correlation_heatmap(df_tails, group_col='tail_n', metrics=tail_confidence_metrics)

fig_hist_tails.savefig(f"{results_output_dir}/figures/{model}/tail_n_histograms.png", bbox_inches='tight')
fig_corr_tails.savefig(f"{results_output_dir}/figures/{model}/tail_n_correlation_heatmap.png", bbox_inches='tight')

In [ ]:
df_topk_logprobs = df[(df.group_size == 1024) & (df.tail_n == 2048)][standard_confidence_metrics + ['top_k_logprobs', 'is_correct', 'turn', 'model']]

fig_hist_topk = plot_grouped_histograms_with_js(df_topk_logprobs, group_col='top_k_logprobs')
fig_corr_topk = plot_grouped_correlation_heatmap(df_topk_logprobs, group_col='top_k_logprobs', metrics=standard_confidence_metrics)

fig_hist_topk.savefig(f"{results_output_dir}/figures/{model}/top_k_logprobs_histograms.png", bbox_inches='tight')
fig_corr_topk.savefig(f"{results_output_dir}/figures/{model}/top_k_logprobs_correlation_heatmap.png", bbox_inches='tight')

In [ ]:
df_group_size = df[(df.top_k_logprobs == 10) & (df.tail_n == 2048)][group_bottom_metrics + ['group_size', 'is_correct', 'turn', 'model']]

fig_hist_group_size = plot_grouped_histograms_with_js(df_group_size, group_col='group_size')
fig_corr_group_size = plot_grouped_correlation_heatmap(df_group_size, group_col='group_size', metrics=group_bottom_metrics)

fig_hist_group_size.savefig(f"{results_output_dir}/figures/{model}/group_size_histograms.png", bbox_inches='tight')
fig_corr_group_size.savefig(f"{results_output_dir}/figures/{model}/group_size_correlation_heatmap.png", bbox_inches='tight')

# Subsampling rollouts

In [16]:
import pandas as pd

def subsample_by_problem_and_turn(df, n_samples=64, random_state=None):
    """
    Group the DataFrame by 'problem_id' and 'turn', and randomly subsample
    each group to contain up to n_samples rows.
    Returns a new, independent DataFrame (no shared reference with input).
    """
    if "problem_id" not in df.columns or "turn" not in df.columns:
        raise ValueError("DataFrame must contain 'problem_id' and 'turn' columns.")

    grouped = df.groupby(["problem_id", "turn"], group_keys=False)

    # Perform sampling
    sampled_df = grouped.apply(
        lambda g: g.sample(n=min(len(g), n_samples), random_state=random_state)
    )

    # Return a completely new DataFrame (deep copy, fresh index)
    return sampled_df.reset_index(drop=True).copy(deep=True)


In [17]:
df.columns

Index(['problem_id', 'rollout_idx', 'extracted_answer', 'is_correct',
       'expected_answer', 'total_tokens', 'mean_full', 'mean_tail',
       'mean_group_lowest', 'mean_group_bottom_pct', 'mean_group_highest',
       'mean_group_top_pct', 'median_full', 'median_tail',
       'median_group_lowest', 'median_group_bottom_pct',
       'median_group_highest', 'median_group_top_pct', 'variance_full',
       'variance_tail', 'variance_group_lowest', 'variance_group_bottom_pct',
       'variance_group_highest', 'variance_group_top_pct', 'gap_full',
       'gap_tail', 'gap_group_lowest', 'gap_group_bottom_pct',
       'gap_group_highest', 'gap_group_top_pct', 'entropy_full',
       'entropy_tail', 'entropy_group_lowest', 'entropy_group_bottom_pct',
       'entropy_group_highest', 'entropy_group_top_pct', 'gap_probs_full',
       'gap_probs_tail', 'gap_probs_group_lowest',
       'gap_probs_group_bottom_pct', 'gap_probs_group_highest',
       'gap_probs_group_top_pct', 'model', 'turn', 'top_k

In [18]:
data = df.loc[(df.group_size == 1024) & (df.tail_n == 2048) & (df.top_k_logprobs == 10) ].reset_index(drop=True).copy(deep=True)
len(data)

5760

In [19]:
64*30*3

5760

In [20]:
subsampled_32 = subsample_by_problem_and_turn(data, n_samples=32, random_state=42)
subsampled_16 = subsample_by_problem_and_turn(data, n_samples=16, random_state=42)
subsampled_8 = subsample_by_problem_and_turn(data, n_samples=8, random_state=42)

/tmp/ipykernel_149574/3277674705.py:15: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sampled_df = grouped.apply(
/tmp/ipykernel_149574/3277674705.py:15: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sampled_df = grouped.apply(
/tmp/ipykernel_149574/3277674705.py:15: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping colu

In [21]:
subsampled_32['samples_per_group'] = 32
subsampled_16['samples_per_group'] = 16
subsampled_8['samples_per_group'] = 8
data['samples_per_group'] = 64

In [22]:
data_subsampled = pd.concat([subsampled_32, subsampled_16, subsampled_8, data], ignore_index=True)

In [ ]:
data_to_plot = data_subsampled[standard_confidence_metrics + ['samples_per_group', 'is_correct', 'turn', 'model']]

plot_grouped_histograms_with_js(data_to_plot, group_col='samples_per_group')
plot_grouped_correlation_heatmap(data_to_plot, group_col='samples_per_group', metrics=standard_confidence_metrics)

In [24]:
data_subsampled

,problem_id,rollout_idx,extracted_answer,is_correct,expected_answer,total_tokens,mean_full,mean_tail,mean_group_lowest,mean_group_bottom_pct,...,gap_probs_group_lowest,gap_probs_group_bottom_pct,gap_probs_group_highest,gap_probs_group_top_pct,model,turn,top_k_logprobs,tail_n,group_size,samples_per_group
0,aime_2025_0,52,70,1,70,4085,17.644214,16.274035,9.850756,14.506067,...,0.729095,0.741512,0.999652,0.909166,Qwen8b,1,10,2048,1024,32
1,aime_2025_0,58,70,1,70,7218,17.416321,16.670980,9.850756,15.014390,...,0.744753,0.760783,0.999652,0.907434,Qwen8b,1,10,2048,1024,32
2,aime_2025_0,0,70,1,70,6257,16.742598,14.984667,9.813254,14.609700,...,0.730854,0.749623,0.999652,0.885817,Qwen8b,1,10,2048,1024,32
3,aime_2025_0,44,70,1,70,4196,17.482597,15.471700,9.850756,15.156092,...,0.735208,0.744483,0.999652,0.889387,Qwen8b,1,10,2048,1024,32
4,aime_2025_0,5,70,1,70,5848,17.133210,14.992631,9.888238,13.732951,...,0.758375,0.768601,0.999656,0.902519,Qwen8b,1,10,2048,1024,32
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10795,aime_2025_13,59,287,0,60,32597,14.278154,11.260348,9.178788,10.288134,...,0.678909,0.722416,0.999394,0.922653,Qwen8b,2,10,2048,1024,64
10796,aime_2025_13,60,100,0,60,36979,14.776232,10.661805,9.638451,10.603248,...,0.655446,0.711741,0.999394,0.959789,Qwen8b,2,10,2048,1024,64
10797,aime_2025_13,61,37,0,60,32188,14.409651,10.249954,8.897952,10.137484,...,0.656445,0.688867,0.999264,0.952970,Qwen8b,2,10,2048,1024,64
10798,aime_2025_13,62,507,0,60,39084,14.235436,10.569094,9.604077,10.393741,...,0.708032,0.746922,0.999394,0.971926,Qwen8b,2,10,2048,1024,64


# Strategies

In [25]:
def load_all_strategies(results_dir="results"):
    """
    Iterates through all JSONL files in the results directory, extracts parameters
    from filenames, loads their contents, and returns a combined DataFrame.
    """
    pattern = re.compile(
        r"strategies_(?P<model>.+?)_(?P<turn>\d+)_top(?P<topk>\d+)logprobs_tail(?P<tail>\d+)_group(?P<group>\d+)\.jsonl"
    )

    all_records = []

    for filename in os.listdir(results_dir):
        if not filename.endswith(".jsonl"):
            continue

        match = pattern.match(filename)
        if not match:
            continue  # skip files that don't fit the pattern

        params = match.groupdict()
        file_path = os.path.join(results_dir, filename)

        # Load JSONL file
        try:
            with open(file_path) as f:
                data_strategies = json.load(f)

            strategy_details = data_strategies.get("strategy_results", {})

            for strategy_name, strategy_data in strategy_details.items():
                task_acc = strategy_data.get("accuracy", None)
                if task_acc is not None:
                    all_records.append({
                        'model': params["model"],
                        "turn": int(params["turn"]),
                        "topk": int(params["topk"]),
                        'tail': int(params['tail']),
                        'group_size': int(params['group']),
                        "strategy": strategy_name,
                        "task_accuracy": task_acc,
                    })
        except Exception as e:
            print(f"⚠️ Skipping malformed file: {filename} due to error: {e}")
            continue
        # Add parameters as columns
        
    if not all_records:
        print("No valid JSONL files found.")
        return pd.DataFrame()

    return pd.DataFrame(all_records)

In [26]:
import pandas as pd
import numpy as np

def compute_grouped_strategy_accuracy(df: pd.DataFrame, group_keys=None):
    """
    Compute strategy accuracies (highest/lowest per metric + random + oracle),
    averaged across problems, grouped by chosen keys (e.g., model, turn, group_size).

    Returns a DataFrame with columns:
    [<group_keys>, strategy, accuracy]
    """
    if group_keys is None:
        group_keys = ['model', 'turn', 'group_size']

    # Columns to ignore
    exclude_cols = set(group_keys + [
        'problem_id', 'rollout_idx', 'response_text', 'extracted_answer',
        'expected_answer', 'is_correct', 'total_tokens'
    ])

    # Select numeric metrics only (excluding random/oracle if they exist)
    metric_cols = [
        c for c in df.columns
        if c not in exclude_cols and df[c].dtype in [np.float64, np.int64]
        and c not in ['random', 'oracle']
    ]

    results = []

    # Group by problem_id and chosen grouping keys
    grouped = df.groupby(['problem_id'] + group_keys)

    for (problem_id, *gkeys), group in grouped:
        group_info = {k: v for k, v in zip(group_keys, gkeys)}

        # --- Metric-based strategies ---
        for metric in metric_cols:
            if metric not in group.columns or group[metric].isnull().all():
                continue

            # Highest
            idx_high = group[metric].idxmax()
            is_correct_high = int(group.loc[idx_high, 'is_correct'])

            # Lowest
            idx_low = group[metric].idxmin()
            is_correct_low = int(group.loc[idx_low, 'is_correct'])

            results.append({**group_info, 'strategy': f'highest_{metric}', 'is_correct': is_correct_high})
            results.append({**group_info, 'strategy': f'lowest_{metric}',  'is_correct': is_correct_low})

        # --- Random baseline ---
        idx_rand = np.random.choice(group.index)
        is_correct_rand = int(group.loc[idx_rand, 'is_correct'])
        results.append({**group_info, 'strategy': 'random', 'is_correct': is_correct_rand})

        # --- Oracle baseline ---
        is_correct_oracle = 1 if group['is_correct'].any() else 0
        results.append({**group_info, 'strategy': 'oracle', 'is_correct': is_correct_oracle})

    # Convert to DataFrame
    results_df = pd.DataFrame(results)

    # Average accuracy per group_keys + strategy
    final_df = (
        results_df
        .groupby(group_keys + ['strategy'], as_index=False)['is_correct']
        .mean()
        .rename(columns={'is_correct': 'task_accuracy', 'tail_n': 'n_tail'})
    )

    return final_df


In [27]:
df_strategies_computed = compute_grouped_strategy_accuracy(data_subsampled, group_keys=['turn', 'model', 'samples_per_group', 'group_size', 'tail_n', 'top_k_logprobs'])

In [ ]:
### to create the csv below, run big_analysis_strategies.ipynb !!!
df_strategies_computed.to_csv(f"{results_dir}/computed_strategies_accuracies.csv", index=False)

In [29]:
df_strategies_computed.loc[(df_strategies_computed['strategy'].isin(['lowest_mean_tail', 'highest_mean_tail', 'highest_variance_tail', 'lowest_variance_tail'])) & (df_strategies_computed['n_tail'] == 2048) & (df_strategies_computed['turn'] == 3)  & (df_strategies_computed['samples_per_group'] == 64) ]

,turn,model,samples_per_group,group_size,n_tail,top_k_logprobs,strategy,task_accuracy
837,3,Qwen8b,64,1024,2048,10,highest_mean_tail,0.733333
849,3,Qwen8b,64,1024,2048,10,highest_variance_tail,0.766667
873,3,Qwen8b,64,1024,2048,10,lowest_mean_tail,0.333333
885,3,Qwen8b,64,1024,2048,10,lowest_variance_tail,0.300000


In [30]:
df_strategies = load_all_strategies(results_dir=f"{results_dir}")


In [31]:
df_strategies

,model,turn,topk,tail,group_size,strategy,task_accuracy
0,Qwen8b,5,10,2048,1024,random,0.600000
1,Qwen8b,5,10,2048,1024,oracle,0.866667
2,Qwen8b,5,10,2048,1024,highest_mean_full,0.600000
3,Qwen8b,5,10,2048,1024,lowest_mean_full,0.400000
4,Qwen8b,5,10,2048,1024,highest_mean_tail,0.533333
...,...,...,...,...,...,...,...
4213,Qwen8b,2,6,2048,1024,lowest_gap_probs_group_bottom_pct,0.666667
4214,Qwen8b,2,6,2048,1024,highest_gap_probs_group_highest,0.666667
4215,Qwen8b,2,6,2048,1024,lowest_gap_probs_group_highest,0.633333
4216,Qwen8b,2,6,2048,1024,highest_gap_probs_group_top_pct,0.566667


# Majority voting

In [32]:
df

,problem_id,rollout_idx,extracted_answer,is_correct,expected_answer,total_tokens,mean_full,mean_tail,mean_group_lowest,mean_group_bottom_pct,...,gap_probs_tail,gap_probs_group_lowest,gap_probs_group_bottom_pct,gap_probs_group_highest,gap_probs_group_top_pct,model,turn,top_k_logprobs,tail_n,group_size
0,aime_2025_5,0,504,1,504,4406,16.137308,14.900967,9.422558,13.513722,...,0.827562,0.806519,0.811453,0.999691,0.897176,Qwen8b,1,8,2048,1024
1,aime_2025_5,1,504,1,504,4732,15.954173,14.829415,9.422558,13.347687,...,0.832866,0.785139,0.790034,0.999691,0.889084,Qwen8b,1,8,2048,1024
2,aime_2025_5,2,504,1,504,4496,15.980498,15.806880,9.422558,14.102894,...,0.841437,0.777922,0.784769,0.999691,0.886770,Qwen8b,1,8,2048,1024
3,aime_2025_5,3,504,1,504,5760,16.190179,14.782489,9.422558,13.467209,...,0.839204,0.763011,0.795337,0.999691,0.924077,Qwen8b,1,8,2048,1024
4,aime_2025_5,4,504,1,504,6171,15.324086,13.994901,9.422558,13.055213,...,0.813131,0.738249,0.755648,0.999696,0.885076,Qwen8b,1,8,2048,1024
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
110267,aime_2025_27,59,157,0,248,26066,13.964509,12.123871,9.700938,11.271306,...,0.823863,0.729700,0.767433,0.999398,0.916654,Qwen8b,2,10,1024,1024
110268,aime_2025_27,60,208,0,248,17668,15.577795,14.006580,9.700938,12.037405,...,0.856906,0.736043,0.758181,0.999398,0.919653,Qwen8b,2,10,1024,1024
110269,aime_2025_27,61,7,0,248,40765,13.720530,11.199118,9.538685,10.759916,...,0.861419,0.744042,0.781338,0.999328,0.915483,Qwen8b,2,10,1024,1024
110270,aime_2025_27,62,3,0,248,33214,13.127080,11.142322,9.725934,10.571903,...,0.818741,0.698133,0.770306,0.999441,0.906688,Qwen8b,2,10,1024,1024


In [33]:
def compute_majority_and_weighted_accuracy(data, metrics, strategy='max'):
    results = []

    for (model, turn), df_turn in data.groupby(["model", "turn"]):
        grouped = df_turn.groupby("problem_id")

        # === Plain majority vote ===
        maj_correct = 0
        for pid, dfp in grouped:
            most_common = dfp["extracted_answer"].mode()
            if not most_common.empty:
                maj_answer = most_common.iloc[0]
                expected = dfp["expected_answer"].iloc[0]
                maj_correct += int(maj_answer == expected)
        maj_acc = maj_correct / grouped.ngroups

        # === Weighted voting for each metric ===
        weighted_results = {}
        for metric in metrics:
            correct_count = 0
            for pid, dfp in grouped:
                # Skip if metric column missing or all NaN
                if metric not in dfp.columns or dfp[metric].isna().all():
                    continue

                # Compute weighted vote sums
                weights = dfp.groupby("extracted_answer")[metric].sum()
                if weights.empty:
                    continue
                if strategy == 'max':
                    weighted_answer = weights.idxmax()
                elif strategy == 'min':
                    weighted_answer = weights.idxmin()
                expected = dfp["expected_answer"].iloc[0]
                correct_count += int(weighted_answer == expected)
            #print(f"Model: {model}, Turn: {turn}, Metric: {metric}, Correct: {correct_count}, Total Problems: {grouped.ngroups}")
            weighted_results[metric] = correct_count / grouped.ngroups

        results.append({
            "model": model,
            "turn": turn,
            "majority_vote_acc": maj_acc,
            **weighted_results
        })

    # === Average across turns ===
    df_results = pd.DataFrame(results)
    avg_df = df_results.groupby("model").mean(numeric_only=True).reset_index()
    return avg_df, df_results

def format_results_with_std(df_results):

    # 1️⃣ Select only numeric columns
    numeric_cols = df_results.select_dtypes(include=[np.number]).columns

    # 2️⃣ Compute mean & std per model
    grouped = df_results.groupby("model")[numeric_cols].agg(["mean", "std"])

    # 3️⃣ Round them
    grouped[[(col, "mean") for col in numeric_cols]] = grouped[[(col, "mean") for col in numeric_cols]].round(3)
    grouped[[(col, "std") for col in numeric_cols]] = grouped[[(col, "std") for col in numeric_cols]].round(2)

    # 4️⃣ Combine mean ± std into one column
    # For each numeric column, merge its two subcolumns into a single formatted string
    combined = pd.DataFrame(index=grouped.index)
    for col in numeric_cols:
        combined[col] = grouped[(col, "mean")].astype(str) + " ± " + grouped[(col, "std")].astype(str)

    # 5️⃣ Reset index for clean output
    avg_df = combined.reset_index()

    return avg_df


In [34]:
def compute_weighted_accuracy(data, strategies=['max']):
    summaries = []
    for metrics, type in [(standard_confidence_metrics, 'full'), (tail_confidence_metrics, 'tail'), (group_bottom_metrics, 'group_bottom_pct')]:
        for strat in strategies:
            summary_df, df_results = compute_majority_and_weighted_accuracy(data, metrics, strategy=strat)
            formatted_summary = format_results_with_std(df_results)
            formatted_summary['type'] = type
            
            if type == 'full':
                formatted_summary.columns = [col.replace('_full', '') for col in formatted_summary.columns]
            elif type == 'tail':
                formatted_summary.columns = [col.replace('_tail', '') for col in formatted_summary.columns]
            elif type == 'group_bottom_pct':
                formatted_summary.columns = [col.replace('_group_bottom_pct', '') for col in formatted_summary.columns]
            
            summaries.append((metrics, strat, formatted_summary))
    df_majority_weighted = pd.concat([s[2] for s in summaries], ignore_index=True)
    pivot_df = (
        df_majority_weighted.melt(id_vars=["model", "type"], var_name="metric", value_name="value")
        .pivot_table(index="metric", columns=["model", "type"], values="value", aggfunc="first")
    )
    pivot_df = pivot_df.reindex([
        "majority_vote_acc", "mean", "median", "variance", "gap", "gap_probs",
         "entropy"
    ])
    return pivot_df
            
        



In [35]:
compute_weighted_accuracy(df, strategies=['max'])

model                    Qwen8b                               
type                       full group_bottom_pct          tail
metric                                                        
majority_vote_acc  0.789 ± 0.02     0.789 ± 0.02  0.789 ± 0.02
mean               0.789 ± 0.02     0.789 ± 0.02  0.789 ± 0.02
median             0.789 ± 0.02     0.789 ± 0.02  0.789 ± 0.02
variance           0.789 ± 0.02     0.789 ± 0.02  0.789 ± 0.02
gap                0.789 ± 0.02     0.789 ± 0.02  0.789 ± 0.02
gap_probs          0.789 ± 0.02     0.789 ± 0.02  0.789 ± 0.02
entropy            0.789 ± 0.02        0.8 ± 0.0  0.789 ± 0.02

In [36]:
compute_weighted_accuracy(subsampled_8, strategies=['max'])

model                    Qwen8b                               
type                       full group_bottom_pct          tail
metric                                                        
majority_vote_acc  0.733 ± 0.03     0.733 ± 0.03  0.733 ± 0.03
mean               0.756 ± 0.02     0.756 ± 0.02  0.756 ± 0.02
median             0.756 ± 0.02     0.756 ± 0.02  0.756 ± 0.02
variance           0.756 ± 0.02     0.767 ± 0.03  0.756 ± 0.02
gap                0.744 ± 0.02     0.756 ± 0.02  0.756 ± 0.02
gap_probs          0.744 ± 0.05     0.756 ± 0.02  0.756 ± 0.02
entropy            0.744 ± 0.02     0.744 ± 0.02  0.744 ± 0.05

In [37]:
compute_weighted_accuracy(subsampled_16, strategies=['max'])

model                    Qwen8b                               
type                       full group_bottom_pct          tail
metric                                                        
majority_vote_acc  0.756 ± 0.05     0.756 ± 0.05  0.756 ± 0.05
mean               0.767 ± 0.06     0.767 ± 0.03  0.767 ± 0.03
median             0.767 ± 0.06     0.767 ± 0.03  0.767 ± 0.03
variance           0.756 ± 0.05     0.767 ± 0.03  0.767 ± 0.03
gap                0.767 ± 0.06     0.778 ± 0.04  0.778 ± 0.04
gap_probs          0.767 ± 0.06     0.778 ± 0.04  0.778 ± 0.04
entropy            0.767 ± 0.03     0.767 ± 0.03  0.756 ± 0.05

In [38]:
compute_weighted_accuracy(subsampled_32, strategies=['max'])

model                 Qwen8b                            
type                    full group_bottom_pct       tail
metric                                                  
majority_vote_acc  0.8 ± 0.0        0.8 ± 0.0  0.8 ± 0.0
mean               0.8 ± 0.0        0.8 ± 0.0  0.8 ± 0.0
median             0.8 ± 0.0        0.8 ± 0.0  0.8 ± 0.0
variance           0.8 ± 0.0        0.8 ± 0.0  0.8 ± 0.0
gap                0.8 ± 0.0        0.8 ± 0.0  0.8 ± 0.0
gap_probs          0.8 ± 0.0        0.8 ± 0.0  0.8 ± 0.0
entropy            0.8 ± 0.0     0.811 ± 0.02  0.8 ± 0.0

# Combination of metrics


In [41]:
def create_strategy_selection_matrix(df: pd.DataFrame):
    """
    Create a binary matrix where each row is a rollout and each column is a strategy.
    Values are 1 if that rollout is selected by the strategy, 0 otherwise.
    
    Args:
        df: DataFrame with rollout data
        
    Returns:
        DataFrame with same rows as input, columns for each strategy (highest/lowest for each metric)
    """
    # Group by unique problem instances
    group_keys = ['problem_id', 'turn', 'model']
    
    # Identify metric columns (same logic as original function)
    exclude_cols = set(group_keys + [
        'rollout_idx', 'response_text', 'extracted_answer',
        'expected_answer', 'is_correct', 'total_tokens'
    ])
    metric_cols = [c for c in df.columns if c not in exclude_cols and df[c].dtype in [np.float64, np.int64]]
    #metric_cols = [c for c in metric_cols if 'full' in c]
    
    # Initialize result dataframe with same index as input
    result_df = pd.DataFrame(index=df.index)
    
    # Add identifying columns
    for col in group_keys + ['rollout_idx', 'is_correct']:
        if col in df.columns:
            result_df[col] = df[col]
    
    # Create binary columns for each strategy
    for metric in metric_cols:
        result_df[f"highest_{metric}"] = 0
        result_df[f"lowest_{metric}"] = 0
    
    # Group and mark selected rollouts
    grouped = df.groupby(group_keys)
    
    for group_name, group in grouped:
        for metric in metric_cols:
            # Skip if metric column missing or empty
            if metric not in group.columns or group[metric].isnull().all():
                continue
            
            # Mark highest
            idx_high = group[metric].idxmax()
            result_df.loc[idx_high, f"highest_{metric}"] = 1
            
            # Mark lowest
            idx_low = group[metric].idxmin()
            result_df.loc[idx_low, f"lowest_{metric}"] = 1
    
    return result_df

In [48]:
selection_matrix = create_strategy_selection_matrix(df_tail_2048_k10)

In [49]:
selection_matrix

,problem_id,turn,model,rollout_idx,is_correct,highest_mean_full,lowest_mean_full,highest_mean_tail,lowest_mean_tail,highest_mean_group_lowest,...,highest_gap_probs_group_highest,lowest_gap_probs_group_highest,highest_gap_probs_group_top_pct,lowest_gap_probs_group_top_pct,highest_top_k_logprobs,lowest_top_k_logprobs,highest_tail_n,lowest_tail_n,highest_group_size,lowest_group_size
42496,aime_2025_5,3,Qwen8b,0,1,0,0,0,0,0,...,0,0,0,0,1,1,1,1,1,1
42497,aime_2025_5,3,Qwen8b,1,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
42498,aime_2025_5,3,Qwen8b,2,1,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
42499,aime_2025_5,3,Qwen8b,3,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
42500,aime_2025_5,3,Qwen8b,4,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
104507,aime_2025_13,2,Qwen8b,59,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
104508,aime_2025_13,2,Qwen8b,60,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
104509,aime_2025_13,2,Qwen8b,61,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
104510,aime_2025_13,2,Qwen8b,62,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [50]:
selection_matrix.loc[(selection_matrix['turn'] == 1) & (selection_matrix['problem_id'] == 'aime_2025_5')]

,problem_id,turn,model,rollout_idx,is_correct,highest_mean_full,lowest_mean_full,highest_mean_tail,lowest_mean_tail,highest_mean_group_lowest,...,highest_gap_probs_group_highest,lowest_gap_probs_group_highest,highest_gap_probs_group_top_pct,lowest_gap_probs_group_top_pct,highest_top_k_logprobs,lowest_top_k_logprobs,highest_tail_n,lowest_tail_n,highest_group_size,lowest_group_size
96896,aime_2025_5,1,Qwen8b,0,1,0,0,0,0,1,...,0,0,0,0,1,1,1,1,1,1
96897,aime_2025_5,1,Qwen8b,1,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
96898,aime_2025_5,1,Qwen8b,2,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
96899,aime_2025_5,1,Qwen8b,3,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
96900,aime_2025_5,1,Qwen8b,4,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
96955,aime_2025_5,1,Qwen8b,59,1,0,1,0,1,0,...,0,0,0,0,0,0,0,0,0,0
96956,aime_2025_5,1,Qwen8b,60,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
96957,aime_2025_5,1,Qwen8b,61,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
96958,aime_2025_5,1,Qwen8b,62,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [52]:
full_metrics = [col for col in selection_matrix.columns if 'full' in col and (col.startswith('highest_'))]
tail_metrics = [col for col in selection_matrix.columns if 'tail' in col and (col.startswith('highest_'))]
group_lowest_metrics = [col for col in selection_matrix.columns if 'group_lowest' in col and (col.startswith('highest_'))]
group_highest_metrics = [col for col in selection_matrix.columns if 'group_highest' in col and (col.startswith('highest_'))]
group_bottom_metrics = [col for col in selection_matrix.columns if 'group_bottom_pct' in col and (col.startswith('highest_'))]
group_top_metrics = [col for col in selection_matrix.columns if 'group_top_pct' in  col and (col.startswith('highest_'))]
full_metrics

['highest_mean_full',
 'highest_median_full',
 'highest_variance_full',
 'highest_gap_full',
 'highest_entropy_full',
 'highest_gap_probs_full']

In [53]:
all_metrics_combined = full_metrics + tail_metrics + group_lowest_metrics + group_highest_metrics + group_bottom_metrics + group_top_metrics

In [56]:
selection_matrix["combined_or_full"] = (selection_matrix[full_metrics] != 0).any(axis=1).astype(int)
selection_matrix["combined_or_tail"] = (selection_matrix[tail_metrics] != 0).any(axis=1).astype(int)
selection_matrix["combined_or_group_lowest"] = (selection_matrix[group_lowest_metrics] != 0).any(axis=1).astype(int)
selection_matrix["combined_or_group_highest"] = (selection_matrix[group_highest_metrics] != 0).any(axis=1).astype(int)
selection_matrix["combined_or_group_bottom"] = (selection_matrix[group_bottom_metrics] != 0).any(axis=1).astype(int)
selection_matrix["combined_or_group_top"] = (selection_matrix[group_top_metrics] != 0).any(axis=1).astype(int)

selection_matrix['combined_or_all'] = (selection_matrix[all_metrics_combined] != 0).any(axis=1).astype(int)

In [57]:
selection_matrix['is_correct_combined_or_full'] = selection_matrix['is_correct'] * selection_matrix['combined_or_full']
selection_matrix['is_correct_combined_or_tail'] = selection_matrix['is_correct'] * selection_matrix['combined_or_tail']
selection_matrix['is_correct_combined_or_group_lowest'] = selection_matrix['is_correct'] * selection_matrix['combined_or_group_lowest']
selection_matrix['is_correct_combined_or_group_highest'] = selection_matrix['is_correct'] * selection_matrix['combined_or_group_highest']
selection_matrix['is_correct_combined_or_group_bottom'] = selection_matrix['is_correct'] * selection_matrix['combined_or_group_bottom']
selection_matrix['is_correct_combined_or_group_top'] = selection_matrix['is_correct'] * selection_matrix['combined_or_group_top']

selection_matrix['is_correct_combined_or_all'] = selection_matrix['is_correct'] * selection_matrix['combined_or_all']

In [60]:
df_combined_acc = []
for icc in ['is_correct_combined_or_full',
            'is_correct_combined_or_tail',
            'is_correct_combined_or_group_lowest',
            'is_correct_combined_or_group_highest',
            'is_correct_combined_or_group_bottom',
            'is_correct_combined_or_group_top',
            'is_correct_combined_or_all']:
    df_is_correct_combined = selection_matrix.groupby(['model', 'problem_id', 'turn'])[icc].max().reset_index()
    df_is_correct_combined = df_is_correct_combined.groupby(['model', 'turn'])[icc].mean().reset_index()
    df_is_correct_combined['strategy'] = icc
    df_is_correct_combined = df_is_correct_combined.rename(columns={icc: 'task_accuracy'})
    df_combined_acc.append(df_is_correct_combined)
    
df_combined_acc = pd.concat(df_combined_acc)
df_combined_acc = df_combined_acc.groupby(['model', 'strategy']).agg({'task_accuracy':['mean', 'std']}).reset_index()

In [62]:
df_combined_acc.round(3)

model                              strategy task_accuracy       
                                                         mean    std
0  Qwen8b            is_correct_combined_or_all         0.867  0.000
1  Qwen8b           is_correct_combined_or_full         0.778  0.038
2  Qwen8b   is_correct_combined_or_group_bottom         0.778  0.038
3  Qwen8b  is_correct_combined_or_group_highest         0.778  0.019
4  Qwen8b   is_correct_combined_or_group_lowest         0.811  0.051
5  Qwen8b      is_correct_combined_or_group_top         0.811  0.019
6  Qwen8b           is_correct_combined_or_tail         0.811  0.038